# Alpamayo 2 Super Auto-Labeling

Run this notebook from the repository root with the `Alpamayo 2 Super` kernel. Set `ALPAMAYO2_SUPER_MODEL_ID` to a Hugging Face model id or local release checkpoint before starting the kernel. The notebook selects the validated six-camera/four-frame auto-labeling input profile before model preparation.

In [1]:
import os

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

'expandable_segments:True'

In [2]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from IPython.display import Video, display

from alpamayo2_super import helper
from alpamayo2_super.common.constants import PUBLIC_MODEL_ID
from alpamayo2_super.inference_smoke import resolve_project_path, validate_model_id
from alpamayo2_super.input_profiles import select_task_input
from alpamayo2_super.load_physical_aiavdataset import load_physical_aiavdataset
from alpamayo2_super.models.alpamayo2_super import Alpamayo2Super
from alpamayo2_super.text_tasks import (
    generate_text,
    prepare_text_generation_inputs,
    summarize_auto_labeling_conditioning,
)
from alpamayo2_super.visualization import plot_auto_labeling_result

In [3]:
cwd = Path.cwd()
if (cwd / "examples").exists():
    project_root = cwd
elif (cwd.parent / "examples").exists():
    project_root = cwd.parent
else:
    project_root = cwd

MODEL_ID = os.environ.get("ALPAMAYO2_SUPER_MODEL_ID", PUBLIC_MODEL_ID)
MANIFEST = resolve_project_path(
    os.environ.get("ALPAMAYO2_SUPER_VALIDATION_MANIFEST", "examples/validation_samples.json"),
    project_root,
)
SAMPLE_INDEX = int(os.environ.get("ALPAMAYO2_SUPER_SAMPLE_INDEX", "0"))
MAX_NEW_TOKENS = int(os.environ.get("ALPAMAYO2_SUPER_AUTOLABEL_MAX_NEW_TOKENS", "1024"))
FUTURE_SOURCE = os.environ.get("ALPAMAYO2_SUPER_AUTOLABEL_FUTURE_SOURCE", "ground_truth")
DIFFUSION_STEPS = int(os.environ.get("ALPAMAYO2_SUPER_DIFFUSION_STEPS", "10"))
SEED = int(os.environ.get("ALPAMAYO2_SUPER_SEED", "42"))
OUTPUT_DIR = resolve_project_path(
    os.environ.get("ALPAMAYO2_SUPER_OUTPUT_DIR", "outputs"), project_root
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if FUTURE_SOURCE not in {"ground_truth", "predicted"}:
    raise ValueError("ALPAMAYO2_SUPER_AUTOLABEL_FUTURE_SOURCE must be ground_truth or predicted")

sample = json.loads(MANIFEST.read_text(encoding="utf-8"))["samples"][SAMPLE_INDEX]
clip_id = os.environ.get("ALPAMAYO2_SUPER_CLIP_ID", sample["clip_id"])
t0_us = int(os.environ.get("ALPAMAYO2_SUPER_T0_US", str(sample["t0_us"])))
validate_model_id(MODEL_ID)
if not torch.cuda.is_available():
    raise RuntimeError("Alpamayo 2 Super auto-labeling requires a CUDA GPU.")

print("model:", MODEL_ID)
print("sample:", SAMPLE_INDEX, clip_id, t0_us)
print("future_source:", FUTURE_SOURCE)

model: nvidia/Alpamayo2-Super
sample: 0 030c760c-ae38-49aa-9ad8-f5650a545d26 5100000
future_source: ground_truth


In [4]:
source_data = load_physical_aiavdataset(
    clip_id,
    t0_us=t0_us,
)
data = select_task_input(source_data, "auto_labeling")
conditioning = summarize_auto_labeling_conditioning(data)
print("camera_indices:", data["camera_indices"].tolist())
print("conditioning:\n", json.dumps(conditioning, indent=2))

camera_indices: [0, 1, 2, 3, 5, 6]
conditioning:
 {
  "camera_input": "context_through_t0",
  "camera_frame_count": 4,
  "future_camera_frames_consumed": false,
  "ego_t0_frame_idx": 3,
  "camera_time_offsets_s": [
    -0.332605,
    -0.232611,
    -0.132623,
    -0.03262
  ],
  "history_trajectory_steps": 16,
  "future_trajectory_steps": 64,
  "future_trajectory_time_range_s": [
    0.1,
    6.4
  ]
}


In [5]:
model = Alpamayo2Super.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="cuda:0")

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/15 [00:00<?, ?it/s]

In [6]:
future_xyz = None
future_rot = None
trajectory_cot = ""

if FUTURE_SOURCE == "predicted":
    trajectory_inputs = helper.prepare_model_inputs(data, model.config, model.tokenizer)
    trajectory_inputs = helper.to_device(trajectory_inputs, "cuda")
    torch.cuda.manual_seed_all(SEED)
    with torch.autocast("cuda", dtype=torch.bfloat16):
        pred_xyz, pred_rot, _, extra = model.sample_trajectories_from_data(
            data=trajectory_inputs,
            top_p=0.98,
            temperature=0.6,
            num_traj_samples=1,
            diffusion_kwargs={"inference_step": DIFFUSION_STEPS},
            return_extra=True,
        )
    future_xyz = pred_xyz[:, 0, 0].detach().cpu()
    future_rot = pred_rot[:, 0, 0].detach().cpu()
    trajectory_cot = str(extra["cot"].reshape(-1)[0])
    print("trajectory_cot:\n", trajectory_cot)

In [7]:
task_inputs = prepare_text_generation_inputs(
    data=data,
    model_config=model.config,
    tokenizer=model.tokenizer,
    task="auto_labeling",
    future_xyz=future_xyz,
    future_rot=future_rot,
)
task_inputs = helper.to_device(task_inputs, "cuda")

torch.cuda.manual_seed_all(SEED)
with torch.autocast("cuda", dtype=torch.bfloat16):
    result = generate_text(
        model,
        task_inputs,
        top_p=0.98,
        temperature=0.6,
        max_new_tokens=MAX_NEW_TOKENS,
    )

auto_labeling_text = result["cot_auto_labeling"][0]
auto_labeling_json = result["cot_auto_labeling_json"][0]
print(json.dumps(auto_labeling_json, indent=2))

{
  "critical_components_analysis": "type: roadside construction zone with cones machinery and workers on the right,\nwhy it is critical: cones machinery and workers encroach toward the travel lane within the first 2 seconds requiring extra clearance and speed control.",
  "ego_vehicle_motion_analysis": "1. type: keep lane,\n   duration: 0-2 seconds,\n   motion: proceed straight behind the lead sedan approaching a roadside construction zone on the right.\n2. type: nudge to the left,\n   duration: 2-4 seconds,\n   motion: slight left offset within the lane to create clearance from cones machinery and workers near the curb.\n3. type: keep lane,\n   duration: 4-8 seconds,\n   motion: continue straight while maintaining a safe gap behind the lead vehicle past the work zone and crosswalk.",
  "trajectory_analysis": null,
  "chain_of_causation": "Nudge left to increase clearance from the roadside construction zone"
}


In [8]:
artifact_stem = f"autolabeling_sample{SAMPLE_INDEX}_{clip_id}_{t0_us}_{FUTURE_SOURCE}"
figure_path = OUTPUT_DIR / f"{artifact_stem}_poster.png"
video_path = OUTPUT_DIR / f"{artifact_stem}.mp4"
json_path = OUTPUT_DIR / f"{artifact_stem}.json"
fig, figure_metadata = plot_auto_labeling_result(
    data=data,
    auto_labeling_json=auto_labeling_json,
    auto_labeling_text=auto_labeling_text,
    future_source=FUTURE_SOURCE,
    trajectory_cot=trajectory_cot,
    output_path=figure_path,
    video_path=video_path,
    model_id=MODEL_ID,
    seed=SEED,
)
plt.close(fig)
display(
    Video(
        filename=str(video_path),
        embed=True,
        html_attributes="controls loop autoplay muted",
    )
)
payload = {
    "task": "auto_labeling",
    "model_id": MODEL_ID,
    "clip_id": clip_id,
    "t0_us": t0_us,
    "seed": SEED,
    "future_source": FUTURE_SOURCE,
    "conditioning": conditioning,
    "diffusion_steps": DIFFUSION_STEPS if FUTURE_SOURCE == "predicted" else None,
    "max_new_tokens": MAX_NEW_TOKENS,
    "trajectory_cot": trajectory_cot,
    "auto_labeling_text": auto_labeling_text,
    "auto_labeling_json": auto_labeling_json,
    "raw_output": result["raw_outputs"][0],
    "figure_path": str(figure_path),
    "video_path": str(video_path),
    "figure_metadata": figure_metadata,
}
json_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("saved_poster:", figure_path)
print("saved_mp4:", video_path)
print("saved_json:", json_path)

saved_poster: /mnt/efs/users/rod/repos/alpamayo2/outputs/autolabeling_sample0_030c760c-ae38-49aa-9ad8-f5650a545d26_5100000_ground_truth_poster.png
saved_mp4: /mnt/efs/users/rod/repos/alpamayo2/outputs/autolabeling_sample0_030c760c-ae38-49aa-9ad8-f5650a545d26_5100000_ground_truth.mp4
saved_json: /mnt/efs/users/rod/repos/alpamayo2/outputs/autolabeling_sample0_030c760c-ae38-49aa-9ad8-f5650a545d26_5100000_ground_truth.json
